In [1]:
import pandas as pd
import numpy as np
import json
import os
from os.path import join
from tqdm import tqdm

In [12]:
folder = r'/mnt/c/Users/tre_i/LBNL FDD Datasets/LBNL_FDD_Data_Sets_Boiler_Plant_all_3/LBNL_FDD_Dataset_Boiler_Plant'

In [13]:
files = sorted(os.listdir(folder))
files

['BoilerPlant.csv',
 'BoilerPlant_boiler_PI.csv',
 'BoilerPlant_boiler_bias_-2.csv',
 'BoilerPlant_boiler_bias_-4.csv',
 'BoilerPlant_boiler_bias_2.csv',
 'BoilerPlant_boiler_bias_4.csv',
 'BoilerPlant_boiler_foul_065.csv',
 'BoilerPlant_boiler_foul_080.csv',
 'BoilerPlant_boiler_foul_095.csv',
 'BoilerPlant_hot_water_pressure_bias_-10.csv',
 'BoilerPlant_hot_water_pressure_bias_-20.csv',
 'BoilerPlant_hot_water_pressure_bias_10.csv',
 'BoilerPlant_hot_water_pressure_bias_20.csv',
 'BoilerPlant_hot_water_temp_bias_-2.csv',
 'BoilerPlant_hot_water_temp_bias_-4.csv',
 'BoilerPlant_hot_water_temp_bias_2.csv',
 'BoilerPlant_hot_water_temp_bias_4.csv']

In [14]:
files[0]

'BoilerPlant.csv'

In [15]:
dfs = {}
for i, file in tqdm(enumerate(files)):
    df = pd.read_csv(join(folder, file), index_col='Datetime')
    dfs[file] = df

0it [00:00, ?it/s]

17it [00:53,  3.16s/it]


In [16]:
import pandas as pd
import hashlib

def df_hash(df):
    # 1. Сортируем колонки
    df = df.sort_index(axis=1)

    # 2. Сортируем индекс
    df = df.sort_index()

    # 3. Округляем числа (чтобы убрать микрошум)
    df = df.round(8)

    # 4. Превращаем в байты
    data_bytes = pd.util.hash_pandas_object(df, index=True).values

    # 5. Считаем общий хэш
    return hashlib.md5(data_bytes).hexdigest()


# создаем словарь для хранения хешей

hashes = {}

for name, data in dfs.items():
    h = df_hash(data)
    hashes.setdefault(h, []).append(name)

# выводим группы одинаковых
for group in hashes.values():
    if len(group) > 1:
        print("Идентичные файлы:")
        for g in group:
            print("  ", g)
        print()

In [7]:
files_non_repeating = ['AHU_annual.csv',
 'coi_bias_-2_annual.csv',
 'coi_bias_-4_annual.csv',
 'coi_bias_2_annual.csv',
 'coi_bias_4_annual.csv',
 'coi_leakage_010_annual.csv',
 'coi_stuck_010_annual.csv',
 'coi_stuck_025_annual.csv',
 'coi_stuck_050_annual.csv',
 'coi_stuck_075_annual.csv',
 'damper_stuck_010_annual.csv',
 'damper_stuck_025_annual.csv',
 'damper_stuck_075_annual.csv',
 'damper_stuck_100_annual_short.csv',
 'oa_bias_-2_annual.csv']

In [8]:
dfs = {}
for i, file in tqdm(enumerate(files_non_repeating)):
    df = pd.read_csv(join(folder, file), index_col='Datetime')
    df.index = pd.to_datetime(df.index)
    df['fault_type'] = i
    df['target'] = i
    df = df.set_index("fault_type", append=True)
    df = df.swaplevel(0, 1).sort_index()
    dfs[file] = df

15it [00:25,  1.73s/it]


In [9]:
for k, df in tqdm(dfs.items()):
    dt = df.index.get_level_values(1)   # 1 = Datetime уровень

    if k == 'damper_stuck_100_annual_short.csv':
        assert dt.min() == pd.Timestamp('2018-04-01 01:00:00')
        assert dt.max() == pd.Timestamp('2018-11-01 00:00:00')
        assert df.shape[0] == 214 * 24 * 60 - 60 + 1
    else:
        assert dt.min() == pd.Timestamp('2018-01-01 01:00:00')
        assert dt.max() == pd.Timestamp('2018-12-31 23:59:00')
        assert df.shape[0] == 365 * 24 * 60 - 60

100%|██████████| 15/15 [00:00<00:00, 167.93it/s]


In [10]:
fault_types_dict = {i: file[:file.find('_annual')] for i, file in enumerate(files)}
fault_types_dict[0] = 'no_fault'

In [11]:
with open('fault_types_dict.json', 'w') as f:
    json.dump(fault_types_dict, f)

In [12]:
df_columns = {k: tuple(v.columns) for k, v in dfs.items()}
len(set(list(df_columns.values())))

1

In [13]:
columns = list(set(list(df_columns.values())))[0]
columns

('CHWC_VLV',
 'CHWC_VLV_DM',
 'MA_TEMP',
 'OA_CFM',
 'OA_DMPR',
 'OA_DMPR_DM',
 'OA_TEMP',
 'RA_CFM',
 'RA_DMPR',
 'RA_DMPR_DM',
 'RA_TEMP',
 'RF_CS',
 'RF_SPD',
 'RF_SPD_DM',
 'RF_WAT',
 'SA_CFM',
 'SA_SP',
 'SA_SPSPT',
 'SA_TEMP',
 'SA_TEMPSPT',
 'SF_CS',
 'SF_SPD',
 'SF_SPD_DM',
 'SF_WAT',
 'SYS_CTL',
 'ZONE_TEMP_1',
 'ZONE_TEMP_2',
 'ZONE_TEMP_3',
 'ZONE_TEMP_4',
 'ZONE_TEMP_5',
 'target')

In [14]:
df_column_values = {k: {c: set(list(v[c])) for c in columns} for k, v in tqdm(dfs.items())}

100%|██████████| 15/15 [00:43<00:00,  2.93s/it]


In [15]:
for k, v in df_column_values.items():
    print(k)
    print([(col, list(values)[0]) for col, values in v.items() if len(values) == 1 and col != 'target'])

AHU_annual.csv
[('OA_CFM', 357730.44), ('SA_SPSPT', 1.60746), ('SA_TEMPSPT', 55.184006), ('SF_SPD', 0.9)]
coi_bias_-2_annual.csv
[('OA_CFM', 357730.44), ('SA_SPSPT', -400.25253), ('SA_TEMPSPT', 55.184006), ('SF_SPD', 0.9)]
coi_bias_-4_annual.csv
[('OA_CFM', 357730.44), ('SA_SPSPT', -400.25253), ('SA_TEMPSPT', 55.184006), ('SF_SPD', 0.9)]
coi_bias_2_annual.csv
[('OA_CFM', 357730.44), ('SA_SPSPT', -400.25253), ('SA_TEMPSPT', 55.184006), ('SF_SPD', 0.9)]
coi_bias_4_annual.csv
[('OA_CFM', 357730.44), ('SA_SPSPT', -400.25253), ('SA_TEMPSPT', 55.184006), ('SF_SPD', 0.9)]
coi_leakage_010_annual.csv
[('OA_CFM', 357730.44), ('SA_SPSPT', -400.25253), ('SA_TEMPSPT', 55.184006), ('SF_SPD', 0.9)]
coi_stuck_010_annual.csv
[('CHWC_VLV', 0.1), ('OA_CFM', 357730.44), ('SA_SPSPT', -400.25253), ('SA_TEMPSPT', 55.184006), ('SF_SPD', 0.9)]
coi_stuck_025_annual.csv
[('CHWC_VLV', 0.25), ('OA_CFM', 357730.44), ('SA_SPSPT', -400.25253), ('SA_TEMPSPT', 55.184006), ('SF_SPD', 0.9)]
coi_stuck_050_annual.csv
[('CH

In [16]:
dfs['AHU_annual.csv'].head()

CHWC_VLV  CHWC_VLV_DM    MA_TEMP  \
fault_type Datetime                                                    
0          2018-01-01 01:00:00  2.635303e-21          0.0  66.374680   
           2018-01-01 01:01:00  2.479578e-21          0.0  66.374680   
           2018-01-01 01:02:00  2.380361e-21          0.0  66.374680   
           2018-01-01 01:03:00 -1.274259e-21          0.0  66.374626   
           2018-01-01 01:04:00  2.749987e-21          0.0  66.374626   

                                   OA_CFM  OA_DMPR  OA_DMPR_DM    OA_TEMP  \
fault_type Datetime                                                         
0          2018-01-01 01:00:00  357730.44      0.0         0.0  10.355011   
           2018-01-01 01:01:00  357730.44      0.0         0.0  10.040033   
           2018-01-01 01:02:00  357730.44      0.0         0.0  10.055031   
           2018-01-01 01:03:00  357730.44      0.0         0.0  10.070026   
           2018-01-01 01:04:00  357730.44      0.0         0.0  10.085022   

                                  RA_CFM  RA_DMPR  RA_DMPR_DM  ...  SF_SPD  \
fault_type Datetime                                            ...           
0          2018-01-01 01:00:00  0.853194      1.0         1.0  ...     0.9   
           2018-01-01 01:01:00  0.853057      1.0         1.0  ...     0.9   
           2018-01-01 01:02:00  0.852918      1.0         1.0  ...     0.9   
           2018-01-01 01:03:00  0.852779      1.0         1.0  ...     0.9   
           2018-01-01 01:04:00  0.852641      1.0         1.0  ...     0.9   

                                SF_SPD_DM        SF_WAT  SYS_CTL  ZONE_TEMP_1  \
fault_type Datetime                                                             
0          2018-01-01 01:00:00        0.0 -2.791519e-13      0.0     73.94652   
           2018-01-01 01:01:00        0.0 -2.790612e-13      0.0     73.97777   
           2018-01-01 01:02:00        0.0 -2.789696e-13      0.0     74.00836   
           2018-01-01 01:03:00        0.0 -2.788807e-13      0.0     74.03825   
           2018-01-01 01:04:00        0.0 -2.787895e-13      0.0     74.06741   

                                ZONE_TEMP_2  ZONE_TEMP_3  ZONE_TEMP_4  \
fault_type Datetime                                                     
0          2018-01-01 01:00:00    66.767230    67.027100    66.761345   
           2018-01-01 01:01:00    66.766680    67.025894    66.761290   
           2018-01-01 01:02:00    66.766014    67.024580    66.761185   
           2018-01-01 01:03:00    66.765305    67.023150    66.761020   
           2018-01-01 01:04:00    66.764530    67.021670    66.760740   

                                ZONE_TEMP_5  target  
fault_type Datetime                                  
0          2018-01-01 01:00:00    67.206566       0  
           2018-01-01 01:01:00    67.207720       0  
           2018-01-01 01:02:00    67.208760       0  
           2018-01-01 01:03:00    67.209700       0  
           2018-01-01 01:04:00    67.210570       0  

[5 rows x 31 columns]

In [17]:
features_to_remove = ['OA_CFM', 'SA_SPSPT', 'SA_SP', 'SA_TEMPSPT', 'SF_SPD']

In [18]:
columns_to_keep = list(set(columns) - set(features_to_remove))

In [19]:
dfs_fixed = {k: v[columns_to_keep] for k, v in dfs.items()}

In [20]:
data_full_year = pd.concat([dfs_fixed[file] for file in files_non_repeating if file != 'damper_stuck_100_annual_short.csv'])
data_full_year = data_full_year.sort_index()
data_damper_stuck_100 = dfs_fixed['damper_stuck_100_annual_short.csv']

In [21]:
cut1 = pd.Timestamp("2018-08-01")
cut2 = pd.Timestamp("2018-10-01")

cut_d1 = pd.Timestamp("2018-08-01")
cut_d2 = pd.Timestamp("2018-10-01")

dt_full = data_full_year.index.get_level_values(1)
dt_damper = data_damper_stuck_100.index.get_level_values(1)

train = pd.concat([
    data_full_year[dt_full < cut1],
    data_damper_stuck_100[dt_damper < cut_d1],
]).sort_index()

val = pd.concat([
    data_full_year[(dt_full >= cut1) & (dt_full < cut2)],
    data_damper_stuck_100[(dt_damper >= cut_d1) & (dt_damper < cut_d2)],
]).sort_index()

test = pd.concat([
    data_full_year[dt_full >= cut2],
    data_damper_stuck_100[dt_damper >= cut_d2],
]).sort_index()

In [22]:
train_df = train.drop('target', axis=1)
train_target = train['target']
val_df = val.drop('target', axis=1)
val_target = val['target']
test_df = test.drop('target', axis=1)
test_target = test['target']

In [23]:
write_to = r'/home/ilya_treyvish/projects/lbnl_fdd/data/processed/SDAHU'

In [24]:
train.shape, val.shape, test.shape

((4448700, 26), (1317600, 26), (1899361, 26))

In [25]:
train_df.to_csv(join(write_to, 'train_df.csv'))
print(len(train_df))
train_target.to_csv(join(write_to, 'train_target.csv'))
print(len(train_target))
val_df.to_csv(join(write_to, 'val_df.csv'))
print(len(val_df))
val_target.to_csv(join(write_to, 'val_target.csv'))
print(len(val_target))
test_df.to_csv(join(write_to, 'test_df.csv'))
print(len(test_df))
test_target.to_csv(join(write_to, 'test_target.csv'))
print(len(test_target))

4448700
4448700
1317600
1317600
1899361
1899361


In [26]:
data = pd.concat([data_full_year, data_damper_stuck_100])

In [27]:
data.to_csv(join(write_to, 'full_year_data.csv'))